In [8]:
from idlelib.multicall import r

import cobra.io

model = cobra.io.load_json_model("iML1515.json")

print(model)
print(len(model.genes))
print(len(model.reactions))
print(len(model.metabolites))

iML1515
1516
2712
1877


In [25]:
import pandas as pd

df = pd.DataFrame([{
    "id": r.id,
    "name": r.name,
    "genes": [g.id for g in r.genes],
    "lower_bound": r.lower_bound,
    "upper_bound": r.upper_bound,
} for r in model.reactions])

#print(df)

genes_of_interest = {"b2048", "b2047"}

mask = df["genes"].apply(lambda genes: any(g in genes_of_interest for g in genes))

filtered_df = df[mask]

print(filtered_df)



          id                                               name    genes  \
276    PMANM                                 Phosphomannomutase  [b2048]   
2674  UDPGPT  Undecaprenyl-phosphate glucose phosphotransferase  [b2047]   

      lower_bound  upper_bound  
276       -1000.0       1000.0  
2674      -1000.0       1000.0  


In [26]:
#RUN FBA

solution = model.optimize()
print(solution.status)
print(model.objective)

rxns_of_interest = {"PMANM", "UDPGPT"}

for rxns in rxns_of_interest:
    print(rxns, solution.fluxes[rxns])

optimal
Maximize
1.0*BIOMASS_Ec_iML1515_core_75p37M - 1.0*BIOMASS_Ec_iML1515_core_75p37M_reverse_35685
UDPGPT 0.0
PMANM 0.0


In [27]:
from cobra.flux_analysis import flux_variability_analysis

fva = flux_variability_analysis(model, reaction_list=["PMANM", "UDPGPT"])
print(fva)

             minimum  maximum
PMANM  -1.590140e-12      0.0
UDPGPT  0.000000e+00      0.0


In [29]:
rxn = model.reactions.get_by_id("PMANM")
print(rxn.reaction)

man1p_c <=> man6p_c


In [30]:
rxn = model.reactions.get_by_id("UDPGPT")
print(rxn.reaction)

udcpp_c + udpg_c <=> udcpgl_c + ump_c


In [31]:
# Identifies the min and max potential flux while maintaining optimal metabolic state
# the range of feasible flux distributions
# UDPGPT is not even used

fva = flux_variability_analysis(
    model,
    reaction_list=["PMANM", "UDPGPT"],
    fraction_of_optimum=0.0 #no enforced minimum flix based on optimal objective value
)
print(fva)

        minimum  maximum
PMANM   -228.14      0.0
UDPGPT     0.00      0.0
